<a href="https://colab.research.google.com/github/angelasidhu/About-Angela/blob/main/qualtrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Steps:
  1. Drop Qualtrics metadata rows (question-text row + import-ID row)
  2. Remove test / preview responses
  3. Filter to consenting respondents only
  4. Drop Qualtrics system columns that are not analytically useful
  5. Convert date columns to proper datetime dtype
  6. Convert Duration to numeric
  7. Reset the index and export to CSV
"""

'\nSteps:\n  1. Drop Qualtrics metadata rows (question-text row + import-ID row)\n  2. Remove test / preview responses\n  3. Filter to consenting respondents only\n  4. Drop Qualtrics system columns that are not analytically useful\n  5. Convert date columns to proper datetime dtype\n  6. Convert Duration to numeric\n  7. Reset the index and export to CSV\n'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# set paths and load file
import pandas as pd
RAW_PATH    = "/content/drive/MyDrive/SmartLink Project 24-26/Qualtrics - Surveys/SmartLink Professionals_April 14, 2026_17.00.csv"
OUTPUT_PATH = "/content/drive/MyDrive/SmartLink Project 24-26/Qualtrics - Surveys/Smartlink_Professionals_clean.csv"

df_raw = pd.read_csv(RAW_PATH, header=0, dtype=str)
print(f"Raw shape: {df_raw.shape}")


Raw shape: (92, 54)


In [ ]:
# drop two header rows
df = df_raw.iloc[2:].copy()
df.reset_index(drop=True, inplace=True)


In [ ]:
# remove test/preview responses
df = df[df["Status"] == "IP Address"]


In [ ]:
# keep only responses with consent
df = df[df["consent"].isin(["I consent", "1"])]
df.reset_index(drop=True, inplace=True)

In [ ]:
TOTAL = len(df)
completed = (df["Finished"] == "True").sum()
partial   = (df["Finished"] == "False").sum()

print("=" * 70)
print("SMARTLINK PROFESSIONALS SURVEY — PERCENTAGE ANALYSIS")
print("=" * 70)
print(f"Total consenting respondents : {TOTAL}")
print(f"  Fully completed            : {completed}")
print(f"  Partial (included below)   : {partial}")
print()
print("Note: each question is analyzed using all answers received for that")
print("question, regardless of whether the respondent finished the survey.")
print("Percentages for multi-select questions sum to >100%.")
print("=" * 70)


SMARTLINK PROFESSIONALS SURVEY — PERCENTAGE ANALYSIS
Total consenting respondents : 88
  Fully completed            : 33
  Partial (included below)   : 55

Note: each question is analyzed using all answers received for that
question, regardless of whether the respondent finished the survey.
Percentages for multi-select questions sum to >100%.


In [ ]:
JUNK = {"1", "2", "3", "4", "5"}   # numeric misfire codes to drop

def print_single(col, label):
    series = df[col].dropna().str.strip()
    series = series[~series.isin(JUNK)]
    n = len(series)
    if n == 0:
        return
    print(f"\n{label}")
    print(f"  (n={n})")
    print("  " + "-" * 50)
    for val, cnt in series.value_counts().items():
        pct = cnt / n * 100
        print(f"  {val:<48}  {pct:5.1f}%  (n={cnt})")

def print_multi(col, label):
    series = df[col].dropna().str.strip()
    n = len(series)
    if n == 0:
        return
    all_choices = series.str.split(",").explode().str.strip()
    all_choices = all_choices[(all_choices != "") & (~all_choices.isin(JUNK))]
    print(f"\n{label}")
    print(f"  (n={n}, multi-select — percentages sum to >100%)")
    print("  " + "-" * 50)
    for val, cnt in all_choices.value_counts().items():
        pct = cnt / n * 100
        print(f"  {val:<48}  {pct:5.1f}%  (n={cnt})")


In [ ]:
def section(title):
    print(f"\n\n{'─' * 70}")
    print(f"  {title.upper()}")
    print(f"{'─' * 70}")


In [ ]:
section("1. Respondent Profile")

print_multi("capacity",
    "Q1. Under what capacity have you worked with folks enrolled in ISAP?")

print_multi("location",
    "Q2. Where are/were your clients located?")

print_multi("Q36",
    "Q3. Where are/were your clients/compas located? (alternate version)")

section("2. Client Demographics")

print_single("number_change",
    "Q4. How much change have you noticed in the number of SmartLink users?")

print_multi("client_status",
    "Q5. What is/was the immigration status of ISAP monitored folks?")

print_multi("client_country",
    "Q6. Country(ies) of origin of ISAP monitored folks")

print_single("minors",
    "Q7. Are/were any of your ISAP monitored folks minors (under 18)?")

section("3. Informed Consent & Transparency")

print_single("govinformed",
    "Q8. Were clients informed that SmartLINK enrollment was not mandatory?")

print_single("BI_support",
    "Q9. Were clients aware that ISAP support was provided by B.I. (a contractor), not DHS?")

print_single("client_algorithm",
    "Q10. Were clients aware that SmartLINK runs a risk prediction algorithm?")

print_single("self_algorithm",
    "Q11. Were you (the professional) aware that SmartLINK runs a risk prediction algorithm?")

section("4. App Experience & Access")

print_single("troubleshoot",
    "Q12. How often could clients access troubleshooting support when SmartLINK malfunctioned?")

print_single("navigation",
    "Q13. How often have clients reported difficulties navigating the SmartLINK app?")

print_single("support_language",
    "Q14. Have clients been able to find ISAP support in their native language?")

section("5. Impact on Clients")

print_single("impact_integration",
    "Q15. Impact of ISAP on clients' ability to integrate into their community")

print_single("impact_wellbeing",
    "Q16. Impact of SmartLINK on clients' overall emotional wellbeing")

section("6. Anxiety & Fear of Surveillance")

print_single("anxious_bracelet",
    "Q17. How often have clients felt anxious about the ankle/wrist bracelet malfunctioning?")

print_single("anxious_Smartl",
    "Q18. How often have clients felt anxious about the SmartLink app malfunctioning?")

print_multi("anxious_concerns",
    "Q19. For anxious clients, what was their concern?")

print_single("fearful",
    "Q20. How often have clients reported feeling fearful of being surveilled by the app?")

section("7. Surveillance Concerns")

print_single("tooluse",
    "Q21. Did clients talk about SmartLink as a tool for:")

print_single("call_concern",
    "Q22. Did clients express concern that their phone calls might be recorded?")

print_single("track_concern",
    "Q23. Did clients express concern that their messages were being tracked/recorded?")

print_single("socmedia_concern",
    "Q24. Did clients express concern that their social media was being tracked/recorded?")

print_single("client_examples",
    "Q25. Did clients provide concrete examples of believing they were surveilled?")

section("8. Compliance & Enforcement")

print_single("compliance",
    "Q26. Did clients regularly comply with virtual SmartLink check-ins?")

print_single("enforcement",
    "Q27. Was your client ever subjected to a surprise in-person visit from immigration enforcement?")

print_single("self_concerns",
    "Q28. Do you have professional concerns about SmartLink as an instrument of immigration enforcement?")

print(f"\n\n{'=' * 70}")
print("END OF REPORT")
print("=" * 70)




──────────────────────────────────────────────────────────────────────
  1. RESPONDENT PROFILE
──────────────────────────────────────────────────────────────────────

Q1. Under what capacity have you worked with folks enrolled in ISAP?
  (n=58, multi-select — percentages sum to >100%)
  --------------------------------------------------
  Legal representative/Aide                          58.6%  (n=34)
  Advocate/organizer                                 29.3%  (n=17)
  Other                                              15.5%  (n=9)
  Social worker                                       1.7%  (n=1)
  Translator                                          1.7%  (n=1)

Q2. Where are/were your clients located?
  (n=58, multi-select — percentages sum to >100%)
  --------------------------------------------------
  Other                                              53.4%  (n=31)
  New York                                           17.2%  (n=10)
  San Francisco                                 

In [ ]:
# resetting index and saving
df.reset_index(drop=True, inplace=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"\nFinal shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Saved to: {OUTPUT_PATH}")



Final shape: 88 rows × 54 columns
Saved to: /content/drive/MyDrive/SmartLink Project 24-26/Qualtrics - Surveys/Smartlink_Professionals_clean.csv
